# Subgroup discovery with a numeric target

With a `NumericTarget`, quality functions compare the distribution of a numeric
attribute in the subgroup, mostly its mean or median, with the whole dataset.
Which groups in the **Adult** census data work the longest hours per week?
The measures and their optimistic estimates follow Lemmerich, Atzmueller, and
Puppe: *Fast exhaustive subgroup discovery with numerical target concepts*
(DMKD 30, 2016). See `01_subgroup_discovery.ipynb` for the basics.

In [ ]:
from sklearn.datasets import fetch_openml

# Adult / Census Income (OpenML dataset 1590): 48,842 people, income above 50K or not.
adult = fetch_openml("adult", version=2, as_frame=True)
# fnlwgt is a census sampling weight, not a property of the person.
data = adult.frame.drop(columns=["fnlwgt"])
features = data.drop(columns="class")
data.head()

In [ ]:
import pandas as pd

import pysubgroup as sd

conditions = sd.generate_conditions(features, n_bins=5)
hours = sd.NumericTarget().fit(data, "hours-per-week")
print(f"overall mean: {hours.values_.mean():.2f} hours")

## Mean-based quality functions

`StandardQFNumeric(a)` uses `size**a * (mean_sg - mean)`: as for binary targets,
`a` trades the size off against the deviation. `centroid="median"` compares
medians instead, and the t-score variant normalizes by the subgroup's standard
deviation. The target attribute itself is excluded from the descriptions
automatically.

In [ ]:
tables = []
for name, qf in {
    "mean (a=0.5)": sd.StandardQFNumeric(0.5),
    "median (a=0.5)": sd.StandardQFNumeric(0.5, centroid="median"),
    "t-score": sd.StandardQFNumericTscore(),
}.items():
    task = sd.SubgroupDiscoveryTask(
        data, hours, conditions, qf, result_set_size=4, depth=2,
        filters=[sd.MinSizeFilter(100)],
    )
    found = sd.DFS().execute(task)
    table = sd.to_dataframe(found, data, hours, ["size", "mean", "median"]).round(4)
    tables.append(table.assign(measure=name))
pd.concat(tables).set_index(["measure", "subgroup"])

`direction="negative"` searches for a decrease instead, here the groups with
the shortest working hours:

In [ ]:
task = sd.SubgroupDiscoveryTask(
    data, hours, conditions, sd.StandardQFNumeric(0.5, direction="negative"),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(100)],
)
shortest = sd.DFS().execute(task)
sd.to_dataframe(shortest, data, hours, ["size", "mean", "mean_difference"]).round(4)

## Other measures: variance, ranks, and distributions

- `VarianceReductionQF` rewards any shift of the mean, up or down (as in
  regression trees).
- `MannWhitneyQF` and `RankAUCQF` use ranks, so outliers matter less.
- `KolmogorovSmirnovQF` rewards any change of the distribution.

In [ ]:
tables = []
for name, qf in {
    "variance reduction": sd.VarianceReductionQF(),
    "Mann-Whitney": sd.MannWhitneyQF(),
    "Kolmogorov-Smirnov": sd.KolmogorovSmirnovQF(),
}.items():
    task = sd.SubgroupDiscoveryTask(
        data, hours, conditions, qf, result_set_size=3, depth=2,
        filters=[sd.MinSizeFilter(100)],
    )
    found = sd.DFS().execute(task)
    table = sd.to_dataframe(found, data, hours, ["size", "mean", "median", "std"]).round(4)
    tables.append(table.assign(measure=name))
pd.concat(tables).set_index(["measure", "subgroup"])

## NumBSD

`NumBSD` (Lemmerich et al. 2016) sorts the rows once by the target and keeps
every condition as a bitset in that order. It computes the tight
ordering-based optimistic estimates in one pass over a subgroup's rows. It
finds the same subgroups as DFS:

In [ ]:
import time

task = sd.SubgroupDiscoveryTask(
    data, hours, conditions, sd.StandardQFNumeric(0.5), result_set_size=10, depth=3,
    filters=[sd.MinSizeFilter(100)],
)
for name, algorithm in {"DFS": sd.DFS(), "NumBSD": sd.NumBSD()}.items():
    started = time.perf_counter()
    found = algorithm.execute(task)
    print(f"{name}: {time.perf_counter() - started:.2f} s, best quality {found[0][0]:.2f}")

## Looking at the subgroups

`plot_target_distributions` (needs matplotlib) compares the target's
distribution in each subgroup with the dataset, and `subgroup_data` returns the
rows of a single subgroup:

In [ ]:
from pysubgroup import visualization as viz

viz.plot_target_distributions(found[:5], data, hours);

In [ ]:
sd.subgroup_data(found[0], data)["hours-per-week"].describe()